# 🏠 German Rental Price Classification

**Dataset:** `immo_data.csv` (German rental listings scraped from ImmoScout24)
**Task:** Multi-class classification
**Method:** CRISP-DM (see `PROJECT_ROADMAP.md`)

---

## 1. Business Understanding

### 1.1 Background
Rent prices in Germany vary widely between regions, cities and types of property. A tenant browsing hundreds of listings finds it hard to judge whether a specific offer is **cheap, fair, or expensive** for what it offers, and a rental platform has no automatic way to tell them.

### 1.2 Problem (draft)
- **Main user:** a rental platform (e.g., ImmoScout24) and, through it, the tenants who use it.
- **What the model does:** predict the **expected** rent class (Low / Medium / High per m²) of a flat from its attributes (location, size, age, condition, equipment…), **without seeing its price**.
- **Decision it supports:** the platform compares the predicted class with the flat's **actual** price:
  - Predicted *Medium*, actually *High* → ⚠️ **possibly overpriced**
  - Predicted *High*, actually *Medium* → 💎 **good deal**

  This helps tenants judge whether an offer is fair.
- **Secondary goal:** find **which attributes drive** a flat into a higher price class.

### 1.3 Target definition
- **Target (decision attribute):** `rentClass` ∈ {Low, Medium, High}
- **Built from:** `rentPerSqm = baseRent ÷ livingSpace`
- **Why per m² and not total rent?** Total rent mainly reflects the flat's **size**. Rent per m² removes the size effect and shows the true **price level**, so small and large flats can be compared fairly.

  | Flat | Size | Total rent | Rent per m² |
  |---|---|---|---|
  | A | 30 m² | €600 | **€20/m²** (expensive) |
  | B | 120 m² | €1,200 | **€10/m²** (cheaper) |

- **How the classes are made:** sort all flats by rent per m² and cut the list into **3 equal-sized groups** (tertiles). Exact cut-off values will be computed in Phase 5.
- **Leakage warning:** since the target is built from rent, every rent-related column (`baseRent`, `totalRent`, `baseRentRange`, `serviceCharge`…) must be **removed from the model inputs**. Otherwise the model would be reading the answer.

### 1.4 Success criteria (draft, to be verified in Phase 5.4)
| Item | Value | Reasoning |
|---|---|---|
| **Assumption** | Classes ≈ balanced (~33% each) | The classes are created by splitting into 3 equal groups |
| **Baseline** | ≈ 33% accuracy | A "no-brain" model that always predicts one class gets 1 in 3 right |
| **Main goal** | **Accuracy ≥ 70%** | Clearly beats the baseline; "good" range for a balanced 3-class problem |
| **Mistake priority** | **Far mistakes (Low ↔ High) must be rare** | The classes are *ordered*: a near mistake (High↔Medium) gives a mild misjudgement, but a far mistake (High↔Low) would put a strongly wrong "overpriced" or "good deal" label on a flat and mislead tenants |
| **Also reported** | Precision / recall / F1 per class, balanced accuracy, confusion matrix | To make sure no single class is predicted badly |

> **Why accuracy is acceptable here:** it is a valid main measure only when (1) the classes are balanced **and** (2) mistakes cost about the same. (1) is true by design. (2) is only partly true, so we add a confusion-matrix check on far mistakes.
>
> If Phase 5.4 shows the classes are *not* balanced, switch the main goal to **balanced accuracy / macro F1**.


In [3]:
import sys

print(sys.executable)

c:\Users\hanan\Desktop\dataSetproject1\.venv\Scripts\python.exe


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


## 2. Data Understanding

In [7]:
path = "../data/raw/immo_data.csv"
df=pd.read_csv(path,low_memory=False)
pd.set_option('display.max_columns', None)
print(df.shape)
print(df.info())
display(df.head(5))



(268850, 49)
<class 'pandas.DataFrame'>
RangeIndex: 268850 entries, 0 to 268849
Data columns (total 49 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   regio1                    268850 non-null  str    
 1   serviceCharge             261941 non-null  float64
 2   heatingType               223994 non-null  str    
 3   telekomTvOffer            236231 non-null  str    
 4   telekomHybridUploadSpeed  45020 non-null   float64
 5   newlyConst                268850 non-null  bool   
 6   balcony                   268850 non-null  bool   
 7   picturecount              268850 non-null  int64  
 8   pricetrend                267018 non-null  float64
 9   telekomUploadSpeed        235492 non-null  float64
 10  totalRent                 228333 non-null  float64
 11  yearConstructed           211805 non-null  float64
 12  scoutId                   268850 non-null  int64  
 13  noParkSpaces              93052 non-null  

,regio1,serviceCharge,heatingType,telekomTvOffer,telekomHybridUploadSpeed,newlyConst,balcony,picturecount,pricetrend,telekomUploadSpeed,totalRent,yearConstructed,scoutId,noParkSpaces,firingTypes,hasKitchen,geo_bln,cellar,yearConstructedRange,baseRent,houseNumber,livingSpace,geo_krs,condition,interiorQual,petsAllowed,street,streetPlain,lift,baseRentRange,typeOfFlat,geo_plz,noRooms,thermalChar,floor,numberOfFloors,noRoomsRange,garden,livingSpaceRange,regio2,regio3,description,facilities,heatingCosts,energyEfficiencyClass,lastRefurbish,electricityBasePrice,electricityKwhPrice,date
0,Nordrhein_Westfalen,245.00,central_heating,ONE_YEAR_FREE,NaN,False,False,6,4.62,10.0,840.0,1965.0,96107057,1.0,oil,False,Nordrhein_Westfalen,True,2.0,595.0,244,86.00,Dortmund,well_kept,normal,NaN,Sch&uuml;ruferstra&szlig;e,Schüruferstraße,False,4,ground_floor,44269,4.0,181.4,1.0,3.0,4,True,4,Dortmund,Schüren,Die ebenerdig zu erreichende Erdgeschosswohnun...,Die Wohnung ist mit Laminat ausgelegt. Das Bad...,NaN,NaN,NaN,NaN,NaN,May19
1,Rheinland_Pfalz,134.00,self_contained_central_heating,ONE_YEAR_FREE,NaN,False,True,8,3.47,10.0,NaN,1871.0,111378734,2.0,gas,False,Rheinland_Pfalz,False,1.0,800.0,NaN,89.00,Rhein_Pfalz_Kreis,refurbished,normal,no,no_information,NaN,False,5,ground_floor,67459,3.0,NaN,NaN,NaN,3,False,4,Rhein_Pfalz_Kreis,Böhl_Iggelheim,Alles neu macht der Mai – so kann es auch für ...,NaN,NaN,NaN,2019.0,NaN,NaN,May19
2,Sachsen,255.00,floor_heating,ONE_YEAR_FREE,10.0,True,True,8,2.72,2.4,1300.0,2019.0,113147523,1.0,NaN,False,Sachsen,True,9.0,965.0,4,83.80,Dresden,first_time_use,sophisticated,NaN,Turnerweg,Turnerweg,True,6,apartment,1097,3.0,NaN,3.0,4.0,3,False,4,Dresden,Äußere_Neustadt_Antonstadt,Der Neubau entsteht im Herzen der Dresdner Neu...,"* 9 m² Balkon\n* Bad mit bodengleicher Dusche,...",NaN,NaN,NaN,NaN,NaN,Oct19
3,Sachsen,58.15,district_heating,ONE_YEAR_FREE,NaN,False,True,9,1.53,40.0,NaN,1964.0,108890903,NaN,district_heating,False,Sachsen,False,2.0,343.0,35,58.15,Mittelsachsen_Kreis,NaN,NaN,NaN,Gl&uuml;ck-Auf-Stra&szlig;e,Glück-Auf-Straße,False,2,other,9599,3.0,86.0,3.0,NaN,3,False,2,Mittelsachsen_Kreis,Freiberg,Abseits von Lärm und Abgasen in Ihre neue Wohn...,NaN,87.23,NaN,NaN,NaN,NaN,May19
4,Bremen,138.00,self_contained_central_heating,NaN,NaN,False,True,19,2.46,NaN,903.0,1950.0,114751222,NaN,gas,False,Bremen,False,1.0,765.0,10,84.97,Bremen,refurbished,NaN,NaN,Hermann-Henrich-Meier-Allee,Hermann-Henrich-Meier-Allee,False,5,apartment,28213,3.0,188.9,1.0,NaN,3,False,4,Bremen,Neu_Schwachhausen,Es handelt sich hier um ein saniertes Mehrfami...,Diese Wohnung wurde neu saniert und ist wie fo...,NaN,NaN,NaN,NaN,NaN,Feb20


## Data dictionary

In [10]:
summary=pd.DataFrame({"dtype":df.dtypes,
                      "non_null":df.notna().sum(),
                      "missing_%":(df.isna().mean()*100).round(1),
                      "unique":df.nunique(),
                      "example":df.iloc[0]})

summary.sort_values(by="missing_%",ascending=False)

,dtype,non_null,missing_%,unique,example
telekomHybridUploadSpeed,float64,45020,83.3,1,NaN
electricityBasePrice,float64,46846,82.6,2,NaN
electricityKwhPrice,float64,46846,82.6,15,NaN
energyEfficiencyClass,str,77787,71.1,10,NaN
lastRefurbish,float64,80711,70.0,88,NaN
heatingCosts,float64,85518,68.2,5669,NaN
noParkSpaces,float64,93052,65.4,71,1.0
petsAllowed,str,154277,42.6,3,NaN
interiorQual,str,156185,41.9,4,normal
thermalChar,float64,162344,39.6,7847,181.4


### 2.5 Column types

In [11]:
# --- 1. Decisions that need human knowledge -------------------------------
id_address = ["scoutId", "houseNumber", "street", "streetPlain"]  # identify ONE listing/place
free_text  = ["description", "facilities"]                        # long sentences
codes      = ["geo_plz",                                          # postcode: a label, not a quantity
              "baseRentRange", "livingSpaceRange",                # "...Range" = group number (1, 2, 3...)
              "noRoomsRange", "yearConstructedRange"]
date_cols  = ["date"]

# --- 2. A function that decides the group of ONE column -------------------
def column_group(col):
    n = df[col].nunique()                          # number of different values
    if col in id_address: return "ID / address"
    if col in free_text:  return "Text (free)"
    if col in codes:      return "Categorical (code)"
    if col in date_cols:  return "Date"
    if n == 1:            return "Constant"         # same value in every row
    if pd.api.types.is_bool_dtype(df[col]):    return "Boolean"
    if pd.api.types.is_numeric_dtype(df[col]): return "Numeric"
    return "Categorical (text)"                     # anything left = text labels

# --- 3. Apply it to every column and save the result in the summary -------
# adds a new column to the summary DataFrame, called "group", which contains the group of each column
summary["group"] = [column_group(col) for col in summary.index]

# --- 4. Print each group with its columns ----------------------------------
# .groupby("group") creates a new DataFrame for each group, and .groups.items() returns a list of tuples (group_name, column_names)
for group, cols in summary.groupby("group").groups.items():
    print(f"{group} ({len(cols)}): {list(cols)}\n")

Boolean (6): ['newlyConst', 'balcony', 'hasKitchen', 'cellar', 'lift', 'garden']

Categorical (code) (5): ['yearConstructedRange', 'baseRentRange', 'geo_plz', 'noRoomsRange', 'livingSpaceRange']

Categorical (text) (13): ['regio1', 'heatingType', 'telekomTvOffer', 'firingTypes', 'geo_bln', 'geo_krs', 'condition', 'interiorQual', 'petsAllowed', 'typeOfFlat', 'regio2', 'regio3', 'energyEfficiencyClass']

Constant (1): ['telekomHybridUploadSpeed']

Date (1): ['date']

ID / address (4): ['scoutId', 'houseNumber', 'street', 'streetPlain']

Numeric (17): ['serviceCharge', 'picturecount', 'pricetrend', 'telekomUploadSpeed', 'totalRent', 'yearConstructed', 'noParkSpaces', 'baseRent', 'livingSpace', 'noRooms', 'thermalChar', 'floor', 'numberOfFloors', 'heatingCosts', 'lastRefurbish', 'electricityBasePrice', 'electricityKwhPrice']

Text (free) (2): ['description', 'facilities']



### 2.6 Duplicates and redundant columns

In [ ]:
n_dup_rows=df.duplicated().sum()
n_dup_ids=df["scoutId"].duplicated().sum()

# in case you want to remove duplicates, uncomment the following lines:
# before = len(df)
# df = df.drop_duplicates()            # keeps the FIRST copy, removes the rest
# print("Removed:", before - len(df), "duplicate rows")
print("Fully duplicated rows:", n_dup_rows)
print("Duplicated scoutIds:  ", n_dup_ids)

Fully duplicated rows: 0
Duplicated scoutIds:   0


In [13]:
# from head "regio1", "geo_bln" and "regio2", "geo_krs" looks like they are the same, but with different names. Let's check if they are really the same:
pairs=[("regio1", "geo_bln"), ("regio2", "geo_krs")]

for a,b in pairs:
    same=(df[a]==df[b]).mean()
    print(f"{a} vs {b}: identical in {same:.1%} of rows")

regio1 vs geo_bln: identical in 100.0% of rows
regio2 vs geo_krs: identical in 100.0% of rows


### 2.7 Leakage check
Leakage = an input column that contains the answer (the price), directly or indirectly.
Rule: "Could I know this value without knowing the flat's rent?" If not, it must not be a model input.


In [14]:
leakage_cols = {
    "baseRent":      "IS the rent (used to build the target)",
    "totalRent":     "rent + extra costs, so it contains the rent",
    "baseRentRange": "the rent grouped into bands (1 = cheapest ... 9 = most expensive)",
    "serviceCharge": "extra monthly costs set together with the rent, so it is price information",
    "heatingCosts":  "monthly heating cost, also price information",
}

# Print the list so it is documented in the notebook
for col, reason in leakage_cols.items():
    print(f"- {col}: {reason}")

# Save just the names in a list, so we can use it later
leakage_list = list(leakage_cols.keys())
print("\nColumns to exclude from model inputs:", leakage_list)

- baseRent: IS the rent (used to build the target)
- totalRent: rent + extra costs, so it contains the rent
- baseRentRange: the rent grouped into bands (1 = cheapest ... 9 = most expensive)
- serviceCharge: extra monthly costs set together with the rent, so it is price information
- heatingCosts: monthly heating cost, also price information

Columns to exclude from model inputs: ['baseRent', 'totalRent', 'baseRentRange', 'serviceCharge', 'heatingCosts']


### 2.8 Phase 2 summary
- **Size:** 268,850 listings × 49 columns.
- **Duplicates:** none (0 duplicated rows, 0 duplicated scoutIds).
- **Redundant pairs:** `regio1` = `geo_bln` (state) and `regio2` = `geo_krs` (district) are 100% identical, so only one of each is needed.
- **Constant column:** `telekomHybridUploadSpeed` has a single value, so it carries no information.
- **Not usable as-is:** IDs/addresses (`scoutId`, `houseNumber`, `street`, `streetPlain`), free text (`description`, `facilities`).
- **Codes stored as numbers:** `geo_plz` and the `...Range` columns.
- **Leakage:** `baseRent`, `totalRent`, `baseRentRange`, `serviceCharge`, `heatingCosts` must not be model inputs.